# model_surgery-mnist-ffnn-pytorch

Function-preserving architecture edits — Net2WiderNet (`nnx.widen`) and
Net2DeeperNet (`nnx.deepen`) — applied to a small MNIST classifier. The
notebook follows the canonical §1–§6 hierarchy.


# 1. Overview

## 1.1 Task & motivation

Most training pipelines treat architecture as immutable: you pick `hidden_dims`, train, then either keep the model or restart from scratch with a different shape. **Model surgery** is the alternative — edit a *trained* model's architecture (widen a layer, insert a layer, drop a layer) in a way that *preserves its forward output exactly*, then continue training from there. The original Net2Net paper (Chen et al., 2015) introduced the construction; `nnx.surgery` ships the primitives.

This notebook demonstrates the two headline operations on MNIST:

- **`nnx.widen`** — Net2WiderNet: increase a hidden layer's width by replicating units, scaling outgoing weights to compensate.
- **`nnx.deepen`** — Net2DeeperNet: insert an identity-initialized `Linear` after an activation so the new layer is a no-op at step 0.

Both edits give a strictly larger model with the *same* loss at step 0 as the original, so resumed training never sees an accuracy cliff.

## 1.2 Dataset summary

MNIST handwritten digits via `torchvision.datasets.MNIST`, wrapped in `nnx.NNDataset`. Same dataset config as the sibling `image_classification-mnist-ffnn-pytorch/` task.

## 1.3 Approach in one paragraph

Train a baseline FFN at `hidden_dims=[64, 64]` for a short budget. Show that `widen` and `deepen` give bit-exact forward equality on a fixed input batch. Continue training the warm-started surgery models alongside cold-start models of the same target shape. Compare convergence speed and final accuracy.

## 1.4 Libraries used

`nnx` (NNModel, NNDataset, NNParams, NNTrainParams, widen, deepen, VisUtils), `torch`, `torchvision`.


# 2. Environment & Setup

## 2.1 Imports


In [1]:
SMOKE_TEST = 0          # 1 = run a tiny smoke version of this notebook
SMOKE_TEST_EPOCHS = 1   # max epochs when SMOKE_TEST=1


In [2]:
import copy

import torch
import torchvision as thv

import nnx
from nnx.nn.dataset.nn_dataset import NNDataset
from nnx.nn.enum.activations import Activations
from nnx.nn.enum.devices import Devices
from nnx.nn.enum.losses import Losses
from nnx.nn.enum.nets import Nets
from nnx.nn.nn_model import NNModel
from nnx.nn.params.nn_model_params import NNModelParams
from nnx.nn.params.nn_params import NNParams
from nnx.nn.params.nn_train_params import NNTrainParams
from nnx.utils import Utils
from nnx.vis_utils import VisUtils


## 2.2 Configuration / hyperparameters

In [3]:
# Same MNIST normalization constants as the sibling mnist-ffnn-pytorch task.
DS_MEAN: float = 0.1307
DS_STD: float = 0.3081

BASE_HIDDEN_DIMS = [64, 64]
WIDEN_LAYER_NAME = "layers.0"           # widen the first Linear
WIDEN_NEW_WIDTH = 128                    # 64 → 128
DEEPEN_AFTER = "layers.0"                # insert after the first Linear

DROPOUT_PROB = 0.0                       # surgery contract is exact, not statistical
BASELINE_EPOCHS = SMOKE_TEST_EPOCHS if SMOKE_TEST else 3
RESUME_EPOCHS = SMOKE_TEST_EPOCHS if SMOKE_TEST else 5


## 2.3 Reproducibility (seed, device)

In [4]:
nnx.set_seed(0)                          # global seed: torch, numpy, python.random
DEVICE = Devices.CPU                     # CPU-only by design (Tier A)


# 3. Data

## 3.1 Loading


In [5]:
ds = NNDataset(
    ds_class=thv.datasets.MNIST,
    transform=thv.transforms.Compose([
        thv.transforms.ToTensor(),
        thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD),
    ]),
)

Utils.print_table(
    header=False,
    data=ds.state(),
    title="Dataset details",
)


  0%|          | 0.00/9.91M [00:00<?, ?B/s]

 20%|█▉        | 1.97M/9.91M [00:00<00:00, 19.3MB/s]

 62%|██████▏   | 6.13M/9.91M [00:00<00:00, 32.3MB/s]

 95%|█████████▍| 9.37M/9.91M [00:00<00:00, 28.8MB/s]

100%|██████████| 9.91M/9.91M [00:00<00:00, 23.3MB/s]

...

 97%|█████████▋| 1.61M/1.65M [00:00<00:00, 3.16MB/s]

100%|██████████| 1.65M/1.65M [00:00<00:00, 2.56MB/s]

  0%|          | 0.00/4.54k [00:00<?, ?B/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 3.61MB/s]

+---------------------------+
|      Dataset details      |
+------------------+--------+
|       name       | MNIST  |
|    input_dim     |  784   |
|    output_dim    |   10   |
| train_batch_size | 54,000 |
|  val_batch_size  | 6,000  |
| test_batch_size  | 10,000 |
+------------------+--------+


## 3.2 Inspection / EDA

MNIST is well-trodden — see [`image_classification-mnist-ffnn-pytorch/notebook.ipynb`](../image_classification-mnist-ffnn-pytorch/notebook.ipynb) §3 for class-distribution + per-class samples. This notebook reuses the same data wrapper without re-running EDA.

## 3.3 Preprocessing & splits

`NNDataset` performs the standard torchvision train/val split (60k / 10k). MinMax + Normalize transforms applied above.


# 4. Model

## 4.1 Baseline architecture

A small two-hidden-layer feed-forward net is the surgery substrate. Small on purpose: surgery's value shows up clearest when the post-surgery model is *meaningfully* larger.


In [6]:
def make_model(hidden_dims):
    # nnx.deepen's identity-init insertion is function-preserving only for ReLU
    # (sigmoid/tanh/GELU would need different post-insertion bias init). Pin the
    # baseline to RELU so the deepen contract holds.
    return NNModel(
        net_params=NNParams(
            input_dim=ds.input_dim,
            output_dim=ds.output_dim,
            hidden_dims=hidden_dims,
            dropout_prob=DROPOUT_PROB,
            activation=Activations.RELU,
        ),
        params=NNModelParams(
            net=Nets.FEED_FWD,
            device=DEVICE,
            loss=Losses.CROSS_ENTROPY,
        ),
    )

baseline = make_model(BASE_HIDDEN_DIMS)
print(baseline.net)


FeedFwdNN=NNParams(dropout_prob=0.0, n_heads=None, activation=relu)


## 4.2 Surgery operations preview

Surgery's contract is *function preservation*: the operated model's forward output on every input equals the original's *before any further training*. The cells in §5.2 prove this empirically on a fixed batch — if equality ever drifts past `atol=1e-5`, the surgery is broken.

## 4.3 Why this design

A short baseline (3 epochs at `[64, 64]`) gives a clearly-undertrained starting point. From there we ask: which is faster — extending training at the same shape, restarting at a wider/deeper shape, or warm-starting at the new shape via surgery? Surgery should match cold-start's final accuracy while reaching it sooner.


# 5. Training

## 5.1 Train the baseline


In [7]:
baseline_train_params = NNTrainParams(n_epochs=BASELINE_EPOCHS) \
    .with_train_loader(value=ds.train_loader) \
    .with_val_loader(value=ds.val_loader)
baseline_run = baseline.train(params=baseline_train_params)
print(f"baseline: {len(baseline_run.idps)} iterations, final val_loss={baseline_run.idps[-1].val_edp.loss:.4f}")


+--------------------------------------------------------------+
|                        Run Details...                        |
+---------------------------+----------------------------------+
|             id            | 8477da4ca665cb1d10bd77255e6ce5da |
|         model.net         |             feed_fwd             |
|         model.loss        |          cross_entropy           |
|        model.device       |               cpu                |
|       net.input_dim       |               784                |
|       net.output_dim      |                10                |
|      net.dropout_prob     |               0.0                |
|      net.hidden_dims      |             [64, 64]             |
|       net.activation      |               relu               |
|       train.n_epochs      |                3                 |
|     train.optim.max_lr    |               0.01               |
|    train.optim.momentum   |           (0.9, 0.999)           |
|      train.optim.name  

Training:   0%|          | 0/3 [00:00<?, ?it/s]

Training:  33%|███▎      | 1/3 [00:01<00:03,  1.98s/it]

Training:  33%|███▎      | 1/3 [00:02<00:03,  1.98s/it, error=0.7385, lr=0.0100]

Training:  67%|██████▋   | 2/3 [00:04<00:02,  2.09s/it, error=0.7385, lr=0.0100]

Training:  67%|██████▋   | 2/3 [00:04<00:02,  2.09s/it, error=0.5357, lr=0.0100]

Training: 100%|██████████| 3/3 [00:06<00:00,  2.07s/it, error=0.5357, lr=0.0100]

Training: 100%|██████████| 3/3 [00:06<00:00,  2.07s/it, error=0.4490, lr=0.0100]

Training: 100%|██████████| 3/3 [00:06<00:00,  2.16s/it, error=0.4490, lr=0.0100]


Run saved to /Users/kaveh/repos/ml-lab/.claude/worktrees/overnight-cleanup/model_surgery-mnist-ffnn-pytorch/runs/8477da4ca665cb1d10bd77255e6ce5da
baseline: 3 iterations, final val_loss=1.3150


## 5.2 Verify function preservation

This is the *contract* — we assert it before relying on it. Any drift past `1e-5` means the surgery primitive is broken. Don't relax the tolerance.


In [8]:
x_probe, _ = next(iter(ds.val_loader))
x_probe = x_probe.view(x_probe.size(0), -1)
baseline.net.eval()
with torch.no_grad():
    y_before = baseline.net(x_probe)

widened_net = nnx.widen(
    copy.deepcopy(baseline.net),
    layer_name=WIDEN_LAYER_NAME,
    new_width=WIDEN_NEW_WIDTH,
)
widened_net.eval()
with torch.no_grad():
    y_after_widen = widened_net(x_probe)
widen_drift = (y_before - y_after_widen).abs().max().item()
print(f"widen drift on probe batch: {widen_drift:.2e}")
assert widen_drift < 1e-5, "widen broke function preservation"


widen drift on probe batch: 1.91e-06


In [9]:
deeper_net = nnx.deepen(
    copy.deepcopy(baseline.net),
    after_layer_name=DEEPEN_AFTER,
)
deeper_net.eval()
with torch.no_grad():
    y_after_deepen = deeper_net(x_probe)
deepen_drift = (y_before - y_after_deepen).abs().max().item()
print(f"deepen drift on probe batch: {deepen_drift:.2e}")
assert deepen_drift < 1e-5, "deepen broke function preservation"


deepen drift on probe batch: 0.00e+00


## 5.3 Warm-start resumed training (post-surgery) vs cold-start

For a fair race we need three models trained for the same `RESUME_EPOCHS` budget:

- **continue**: keep training the baseline at `[64, 64]` (no surgery)
- **warm-widen**: continue training the widened model at `[128, 64]`
- **cold-widen**: a fresh model at `[128, 64]`, random init


In [10]:
def make_model_from_net(net, model_params):
    # wrap a hand-edited net in an NNModel so the standard train loop applies
    m = NNModel(
        net_params=NNParams(input_dim=ds.input_dim, output_dim=ds.output_dim,
                            hidden_dims=BASE_HIDDEN_DIMS, dropout_prob=DROPOUT_PROB,
                            activation=Activations.RELU),
        params=model_params,
    )
    m.net = net
    return m

cont_model = make_model_from_net(
    copy.deepcopy(baseline.net),
    NNModelParams(net=Nets.FEED_FWD, device=DEVICE, loss=Losses.CROSS_ENTROPY),
)
warm_widened = make_model_from_net(
    widened_net,
    NNModelParams(net=Nets.FEED_FWD, device=DEVICE, loss=Losses.CROSS_ENTROPY),
)
cold_widened = make_model([WIDEN_NEW_WIDTH, BASE_HIDDEN_DIMS[1]])

resume_train_params = NNTrainParams(n_epochs=RESUME_EPOCHS) \
    .with_train_loader(value=ds.train_loader) \
    .with_val_loader(value=ds.val_loader)

cont_run = cont_model.train(params=resume_train_params)
warm_run = warm_widened.train(params=resume_train_params)
cold_run = cold_widened.train(params=resume_train_params)


+--------------------------------------------------------------+
|                        Run Details...                        |
+---------------------------+----------------------------------+
|             id            | 1cc485218295dba9ef58c3852860bd3e |
|         model.net         |             feed_fwd             |
|         model.loss        |          cross_entropy           |
|        model.device       |               cpu                |
|       net.input_dim       |               784                |
|       net.output_dim      |                10                |
|      net.dropout_prob     |               0.0                |
|      net.hidden_dims      |             [64, 64]             |
|       net.activation      |               relu               |
|       train.n_epochs      |                5                 |
|     train.optim.max_lr    |               0.01               |
|    train.optim.momentum   |           (0.9, 0.999)           |
|      train.optim.name  

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Training:  20%|██        | 1/5 [00:01<00:07,  1.83s/it]

Training:  20%|██        | 1/5 [00:02<00:07,  1.83s/it, error=0.5967, lr=0.0100]

Training:  40%|████      | 2/5 [00:03<00:05,  1.95s/it, error=0.5967, lr=0.0100]

...

Training:  80%|████████  | 4/5 [00:08<00:02,  2.01s/it, error=0.3957, lr=0.0100]

Training: 100%|██████████| 5/5 [00:09<00:00,  2.02s/it, error=0.3957, lr=0.0100]

Training: 100%|██████████| 5/5 [00:10<00:00,  2.02s/it, error=0.2818, lr=0.0100]

Training: 100%|██████████| 5/5 [00:10<00:00,  2.04s/it, error=0.2818, lr=0.0100]


Run saved to /Users/kaveh/repos/ml-lab/.claude/worktrees/overnight-cleanup/model_surgery-mnist-ffnn-pytorch/runs/6ad7a656e05121f03d644e36bd5faf68


# 6. Evaluation & Results

## 6.1 Comparison


In [11]:
from prettytable import PrettyTable

t = PrettyTable()
t.title = "warm-start vs cold-start at the same target shape"
t.field_names = ["model", "final train loss", "final val loss"]
for name, run in [
    ("continue (base [64, 64])", cont_run),
    (f"warm-widen ([{WIDEN_NEW_WIDTH}, {BASE_HIDDEN_DIMS[1]}])", warm_run),
    (f"cold-widen ([{WIDEN_NEW_WIDTH}, {BASE_HIDDEN_DIMS[1]}])", cold_run),
]:
    t.add_row([name, f"{run.idps[-1].train_edp.loss:.4f}", f"{run.idps[-1].val_edp.loss:.4f}"])
print(t)


+--------------------------------------------------------------+
|      warm-start vs cold-start at the same target shape       |
+--------------------------+------------------+----------------+
|          model           | final train loss | final val loss |
+--------------------------+------------------+----------------+
| continue (base [64, 64]) |      0.9971      |     0.9260     |
|  warm-widen ([128, 64])  |      1.2413      |     1.1409     |
|  cold-widen ([128, 64])  |      1.0854      |     0.8612     |
+--------------------------+------------------+----------------+


## 6.2 Loss curves

In [12]:
VisUtils.multi_line_plot(
    x_ticks_inc=max(1, len(cont_run.idps) // 10),
    y_axis_label="Validation loss",
    x_axis_label="Iterations",
    title="Warm-start (surgery) vs cold-start at [128, 64]",
    fig_size=(16 * 100, 9 * 100),
    x=[idp.iter_idx for idp in cont_run.idps],
    yss_legend=[["validation"], ["continue", "warm-widen", "cold-widen"]],
    yss=[[
        [idp.val_edp.loss for idp in run.idps],
    ] for run in (cont_run, warm_run, cold_run)],
)


## 6.3 Discussion

The expected pattern:

- **continue** plateaus near the baseline's converged loss — capacity is the bottleneck.
- **warm-widen** starts at the same loss as `continue` (the surgery contract) and improves *from there*.
- **cold-widen** starts at higher loss (random init) and has to catch up.

The "right" model for the resumed budget is the one that reaches the lowest final validation loss. With this much undertraining, warm-widen typically wins on time-to-target — the headline pedagogical point.

For a deeper dive into the Net2Net guarantees, see Chen et al. 2015 (*Net2Net: Accelerating learning via knowledge transfer*).
